# KRR PROJECT FINAL CODE

# SECTION 1: Setup (Blocks 1-3)

## BLOCK 1: Install all dependencies


In [ ]:
# =============================================================================
# BLOCK 1: Install all dependencies (run after every runtime restart)
# =============================================================================

!pip -q install requests beautifulsoup4 lxml spacy neo4j tqdm \
    pageindex faiss-cpu sentence-transformers yfinance \
    pandas matplotlib

!python -m spacy download en_core_web_sm -q

print("All packages installed.")

In [ ]:
# ============================================================
# BLOCK 1: Colab Setup & Install
# ============================================================

# Mount Google Drive so downloaded SEC files persist
from google.colab import drive
drive.mount('/content/drive')

# Install all required packages
!pip -q install requests beautifulsoup4 lxml spacy neo4j tqdm \
    pageindex faiss-cpu sentence-transformers openai

# Download spaCy model
!python -m spacy download en_core_web_sm -q

print("All packages installed")

## BLOCK 2: Environment Configuration (Colab + CreateAI)

In [ ]:
# =============================================================================
# BLOCK 2: Environment Configuration (Colab + CreateAI)
# =============================================================================

import os
import json
import time
from google.colab import userdata, drive

# Mount Google Drive
drive.mount("/content/drive")

# Load credentials from Colab Secrets
CREATEAI_API_URL    = userdata.get("CREATEAI_API_URL")
CREATEAI_API_TOKEN  = userdata.get("CREATEAI_API_TOKEN")
CREATEAI_PROJECT_ID = userdata.get("CREATEAI_PROJECT_ID")

# Validate
missing = [
    name for name, val in {
        "CREATEAI_API_URL":    CREATEAI_API_URL,
        "CREATEAI_API_TOKEN":  CREATEAI_API_TOKEN,
        "CREATEAI_PROJECT_ID": CREATEAI_PROJECT_ID,
    }.items() if not val
]
if missing:
    raise EnvironmentError(
        f"Missing Colab Secrets: {missing}\n"
        f"Add them via the key icon in the left sidebar."
    )

# Neo4j credentials
os.environ["NEO4J_URI"]      = userdata.get("NEO4J_URI")
os.environ["NEO4J_USERNAME"] = userdata.get("NEO4J_USERNAME")
os.environ["NEO4J_PASSWORD"] = userdata.get("NEO4J_PASSWORD")
os.environ["NEO4J_DATABASE"] = userdata.get("NEO4J_DATABASE")

# SEC credentials
os.environ["SEC_USER_AGENT"] = userdata.get("SEC_USER_AGENT")

# Google Drive paths
DATA_DIR        = "/content/drive/MyDrive/sec_bulk"
DOC_MAP_PATH    = "/content/drive/MyDrive/pageindex_doc_map.json"
FAISS_INDEX_DIR = "/content/drive/MyDrive/faiss_index"

os.makedirs(DATA_DIR,        exist_ok=True)
os.makedirs(FAISS_INDEX_DIR, exist_ok=True)

# Doc map utilities
def save_doc_map(path, doc_map):
    with open(path, "w") as f:
        json.dump(doc_map, f, indent=2)

def load_doc_map(path):
    if not os.path.exists(path):
        return {}
    with open(path, "r") as f:
        return json.load(f)

doc_map = load_doc_map(DOC_MAP_PATH)
doc_map.update({
    "AAPL_0000320193-25-000079": "pi-cmof9tavn004i01petztv5hvd",
    "MSFT_0000950170-25-100235": "pi-cmof9thwh004k01peeu3ysxx2",
})
save_doc_map(DOC_MAP_PATH, doc_map)

print("Environment configured.")
print(f"CreateAI endpoint : {CREATEAI_API_URL}")
print(f"Project ID        : {CREATEAI_PROJECT_ID[:8]}... (truncated)")

## BLOCK 3: LLM Setup using ASU CreateAI (corrected)

In [ ]:
# =============================================================================
# BLOCK 3: LLM Setup using ASU CreateAI (corrected)
# =============================================================================

import requests

CREATEAI_HEADERS = {
    "Authorization": f"Bearer {CREATEAI_API_TOKEN}",
    "Content-Type":  "application/json",
}

def llm_chat(prompt: str, temperature: float = 0.0, max_tokens: int = 1024) -> str:
    payload = {
        "request_source": "override_params",
        "project_id":     CREATEAI_PROJECT_ID,
        "query":          prompt,
        "model_provider": "openai",
        "model_name":     "gpt4o",
        "model_params": {
            "temperature":   temperature,
            "system_prompt": "You are a precise financial analyst assistant.",
        },
    }

    response = requests.post(
        f"{CREATEAI_API_URL}/query",
        headers=CREATEAI_HEADERS,
        json=payload,
        timeout=120,
    )
    response.raise_for_status()

    # Response text is directly at response["response"]
    return response.json()["response"].strip()


# Verify connection
test_response = llm_chat("Reply with the single word CONNECTED.")
print(f"LLM connection status: {test_response}")

# SECTION 2: Data & KG (Blocks 4-5)

## BLOCK 4: Neo4j Store + Financial Metric Enrichment

In [ ]:
# =============================================================================
# BLOCK 4: Neo4j Store + Financial Metric Enrichment
# Two sources combined:
#   1. SEC EDGAR XBRL API  — historical data going back to ~2010
#   2. Yahoo Finance        — fills any gaps in recent years
# =============================================================================

!pip -q install neo4j yfinance

import os
import time
import yfinance as yf
import requests
from neo4j import GraphDatabase
from neo4j.exceptions import TransientError

# ------------------------------------------------------------------ Neo4j store
class Neo4jStore:

    def __init__(self):
        self.driver = GraphDatabase.driver(
            os.environ["NEO4J_URI"],
            auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"])
        )
        self.db = os.environ["NEO4J_DATABASE"]

    def run(self, cypher: str, **params):
        with self.driver.session(database=self.db) as session:
            return session.run(cypher, **params).data()

    def init_schema(self):
        self.run("CREATE CONSTRAINT IF NOT EXISTS FOR (f:Filing)          REQUIRE f.accession_no IS UNIQUE")
        self.run("CREATE CONSTRAINT IF NOT EXISTS FOR (e:Entity)          REQUIRE e.name IS UNIQUE")
        self.run("CREATE CONSTRAINT IF NOT EXISTS FOR (c:Company)         REQUIRE c.ticker IS UNIQUE")
        self.run("CREATE CONSTRAINT IF NOT EXISTS FOR (m:FinancialMetric) REQUIRE m.metric_id IS UNIQUE")
        print("Schema constraints verified.")

    def upsert_financial_metric(self, ticker: str, metric: str, year: int, value: float, unit: str = "USD"):
        metric_id = f"{ticker}_{metric}_{year}"
        self.run("""
            MERGE (c:Company {ticker: $ticker})
            MERGE (m:FinancialMetric {metric_id: $metric_id})
              SET m.ticker = $ticker,
                  m.metric = $metric,
                  m.year   = $year,
                  m.value  = $value,
                  m.unit   = $unit
            MERGE (c)-[:HAS_METRIC]->(m)
        """, ticker=ticker, metric=metric, year=year,
             value=value, unit=unit, metric_id=metric_id)

    def query_metric(self, ticker: str, metric: str, year: int):
        results = self.run("""
            MATCH (m:FinancialMetric {ticker: $ticker, metric: $metric, year: $year})
            RETURN m.value AS value, m.unit AS unit
        """, ticker=ticker, metric=metric, year=year)
        return results[0] if results else None

    def query_all_metrics(self, ticker: str):
        return self.run("""
            MATCH (c:Company {ticker: $ticker})-[:HAS_METRIC]->(m:FinancialMetric)
            RETURN m.metric AS metric, m.year AS year, m.value AS value, m.unit AS unit
            ORDER BY m.year DESC, m.metric
        """, ticker=ticker)

    def query_filings_for_ticker(self, ticker: str):
        return self.run("""
            MATCH (c:Company {ticker: $ticker})-[:FILED]->(f:Filing)
            RETURN f.accession_no AS accession_no
        """, ticker=ticker)

    def close(self):
        self.driver.close()


store = Neo4jStore()
store.init_schema()


# ------------------------------------------------------------------ Source 1: SEC XBRL
# Maps XBRL concept names to our internal Neo4j metric keys
XBRL_CONCEPT_MAP = {
    "Revenues":                                "revenue",
    "RevenueFromContractWithCustomerExcludingAssessedTax": "revenue",
    "SalesRevenueNet":                         "revenue",
    "NetIncomeLoss":                           "net_income",
    "GrossProfit":                             "gross_profit",
    "OperatingIncomeLoss":                     "operating_income",
    "EarningsPerShareBasic":                   "eps_basic",
}

def ticker_to_cik_xbrl(ticker: str) -> str:
    """Converts ticker to zero-padded CIK using SEC company tickers JSON."""
    import re
    data = requests.get(
        "https://www.sec.gov/files/company_tickers.json",
        headers={"User-Agent": os.environ["SEC_USER_AGENT"]},
        timeout=20,
    ).json()
    for _, row in data.items():
        if row["ticker"].upper() == ticker.upper():
            return re.sub(r"\D", "", str(row["cik_str"])).zfill(10)
    raise ValueError(f"Ticker not found: {ticker}")


def enrich_from_xbrl(ticker: str, start_year: int = 2015):
    """
    Pulls historical annual financial data from SEC EDGAR XBRL API.
    Covers fiscal years from start_year to present.
    Only keeps 10-K annual filings (form == '10-K').
    """
    try:
        cik  = ticker_to_cik_xbrl(ticker)
        url  = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
        data = requests.get(
            url,
            headers={"User-Agent": os.environ["SEC_USER_AGENT"]},
            timeout=30,
        ).json()

        us_gaap = data.get("facts", {}).get("us-gaap", {})
        count   = 0

        for xbrl_concept, metric_key in XBRL_CONCEPT_MAP.items():
            if xbrl_concept not in us_gaap:
                continue

            units_block = us_gaap[xbrl_concept].get("units", {})

            # Revenue / income / profit are in USD; EPS is in USD/shares
            unit_key = "USD/shares" if "eps" in metric_key else "USD"
            entries  = units_block.get(unit_key, [])

            for entry in entries:
                # Only annual 10-K filings, not quarterly
                if entry.get("form") != "10-K":
                    continue
                # Use the fiscal year end date to determine the year
                end_date = entry.get("end", "")
                if not end_date:
                    continue
                year = int(end_date[:4])
                if year < start_year:
                    continue

                value = entry.get("val")
                if value is None:
                    continue

                store.upsert_financial_metric(ticker, metric_key, year, float(value))
                count += 1

        print(f"  {ticker}: {count} records from SEC XBRL (back to {start_year})")
        time.sleep(0.3)   # respect SEC rate limit

    except Exception as exc:
        print(f"  {ticker}: XBRL enrichment failed — {exc}")


# ------------------------------------------------------------------ Source 2: Yahoo Finance
# Covers the most recent 4 fiscal years and fills any gaps XBRL missed
YFINANCE_METRIC_MAP = {
    "Total Revenue":    "revenue",
    "Net Income":       "net_income",
    "Gross Profit":     "gross_profit",
    "Operating Income": "operating_income",
    "Basic EPS":        "eps_basic",
}

def enrich_from_yfinance(ticker: str):
    try:
        obj    = yf.Ticker(ticker)
        income = obj.income_stmt
        count  = 0

        for col in income.columns:
            year = col.year
            for yf_label, kg_label in YFINANCE_METRIC_MAP.items():
                if yf_label not in income.index:
                    continue
                raw = income.loc[yf_label, col]
                if raw is None:
                    continue
                if isinstance(raw, float) and raw != raw:
                    continue
                store.upsert_financial_metric(ticker, kg_label, year, float(raw))
                count += 1

        print(f"  {ticker}: {count} records from Yahoo Finance (recent 4 years)")

    except Exception as exc:
        print(f"  {ticker}: yfinance enrichment failed — {exc}")


# ------------------------------------------------------------------ Run both sources
TICKERS_TO_ENRICH = ["AAPL", "MSFT", "AMZN", "NVDA"]

print("Pass 1: SEC XBRL historical data (2015 to present)...")
for ticker in TICKERS_TO_ENRICH:
    enrich_from_xbrl(ticker, start_year=2015)

print("\nPass 2: Yahoo Finance recent data (fills gaps)...")
for ticker in TICKERS_TO_ENRICH:
    enrich_from_yfinance(ticker)
    time.sleep(0.4)

# ------------------------------------------------------------------ Spot check
print("\nSample AAPL metrics in Neo4j (all years):")
rows = store.query_all_metrics("AAPL")
for row in rows:
    if row["metric"] == "revenue":
        print(f"  revenue ({row['year']}): ${row['value']:,.0f}")

In [ ]:
# =============================================================================
# BLOCK 4 FIX: Remove duplicate/quarterly XBRL entries
# Keeps only the single largest revenue value per ticker per year,
# which corresponds to the full annual 10-K figure.
# =============================================================================

def fix_duplicate_metrics(ticker: str):
    """
    For each (ticker, metric, year) combination, keeps only the
    maximum value. This removes quarterly sub-totals that slipped
    through the 10-K filter in the XBRL data.
    """
    rows = store.query_all_metrics(ticker)

    # Group by (metric, year) and find the max value
    from collections import defaultdict
    best = defaultdict(lambda: float("-inf"))

    for row in rows:
        key = (row["metric"], row["year"])
        if row["value"] > best[key]:
            best[key] = row["value"]

    # Overwrite each metric node with the correct max value
    count = 0
    for (metric, year), value in best.items():
        store.run("""
            MATCH (m:FinancialMetric {metric_id: $metric_id})
            SET m.value = $value
        """, metric_id=f"{ticker}_{metric}_{year}", value=value)
        count += 1

    print(f"  {ticker}: {count} metric nodes corrected")


print("Fixing duplicate XBRL entries...")
for ticker in TICKERS_TO_ENRICH:
    fix_duplicate_metrics(ticker)

# Verify AAPL revenue is now correct
print("\nAAPL revenue after fix:")
rows = store.query_all_metrics("AAPL")
for row in rows:
    if row["metric"] == "revenue":
        print(f"  revenue ({row['year']}): ${row['value']:,.0f}")

## BLOCK 5: SEC EDGAR Download

In [ ]:
# =============================================================================
# BLOCK 5: SEC EDGAR Download
# Downloads 10-K filings for AAPL, MSFT, AMZN, NVDA from 2020 onwards.
# Files are saved to Google Drive so they persist across Colab sessions.
# If a file already exists on Drive it is skipped automatically.
# =============================================================================

import requests
import re

UA         = os.environ["SEC_USER_AGENT"]
START_YEAR = 2020
TICKERS    = ["AAPL", "MSFT", "AMZN", "NVDA"]


def cik_pad(cik: str) -> str:
    return re.sub(r"\D", "", cik).zfill(10)


def sec_get(url: str):
    for attempt in range(5):
        try:
            r = requests.get(url, headers={"User-Agent": UA}, timeout=20)
            if r.status_code == 200:
                time.sleep(0.2)
                return r
        except Exception:
            time.sleep(1 * (attempt + 1))
    raise RuntimeError(f"Failed to fetch: {url}")


def ticker_to_cik(ticker: str) -> str:
    data = sec_get("https://www.sec.gov/files/company_tickers.json").json()
    for _, row in data.items():
        if row["ticker"].upper() == ticker.upper():
            return cik_pad(str(row["cik_str"]))
    raise ValueError(f"Ticker not found: {ticker}")


def archive_url(cik: str, accession: str, doc: str) -> str:
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{int(cik)}/{accession.replace('-', '')}/{doc}"
    )


def collect_10k_filings(cik: str):
    subs    = sec_get(f"https://data.sec.gov/submissions/CIK{cik}.json").json()
    results = []
    seen    = set()

    def parse_block(block):
        for form, acc, doc, date in zip(
            block["form"],
            block["accessionNumber"],
            block["primaryDocument"],
            block["filingDate"],
        ):
            if form == "10-K" and int(date[:4]) >= START_YEAR and acc not in seen:
                seen.add(acc)
                results.append((acc, doc, date))

    parse_block(subs["filings"]["recent"])
    for file_ref in subs["filings"].get("files", []):
        historical = sec_get("https://data.sec.gov/submissions/" + file_ref["name"]).json()
        parse_block(historical)

    return results


total_downloaded = 0

for ticker in TICKERS:
    print(f"\n{ticker}")
    cik      = ticker_to_cik(ticker)
    filings  = collect_10k_filings(cik)
    print(f"  Found {len(filings)} filings since {START_YEAR}")

    for acc, doc, date in filings:
        dest_path = os.path.join(DATA_DIR, f"{ticker}_{acc}_{doc}")

        if os.path.exists(dest_path):
            print(f"  Skipping {acc} (already on Drive)")
            continue

        try:
            r = sec_get(archive_url(cik, acc, doc))
            with open(dest_path, "wb") as fh:
                fh.write(r.content)
            print(f"  Downloaded {acc} ({date})")
            total_downloaded += 1
        except Exception as exc:
            print(f"  Failed {acc}: {exc}")

print(f"\nDownload complete. New files downloaded: {total_downloaded}")
print(f"Total files on Drive: {len(os.listdir(DATA_DIR))}")

# SECTION 3: Retrieval Systems (Blocks 6-7)

## BLOCK 6: Vanilla RAG Baseline using FAISS + local sentence-transformers embeddings

In [ ]:
# =============================================================================
# BLOCK 6: Vanilla RAG Baseline — FAISS + local sentence-transformers embeddings
# Uses BAAI/bge-large-en-v1.5 locally instead of CreateAI embeddings endpoint.
# Avoids the 403 permissions issue on the embeddings endpoint.
# =============================================================================

!pip -q install faiss-cpu sentence-transformers

import faiss
import numpy as np
import pickle
import warnings
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

CHUNK_SIZE    = 400
CHUNK_OVERLAP = 50
FAISS_K       = 3

FAISS_INDEX_PATH  = os.path.join(FAISS_INDEX_DIR, "index.faiss")
FAISS_CHUNKS_PATH = os.path.join(FAISS_INDEX_DIR, "chunks.pkl")

print("Loading embedding model...")
EMBED_MODEL = SentenceTransformer("BAAI/bge-large-en-v1.5")
print("Embedding model loaded.")


# ------------------------------------------------------------------ text utils
def html_to_plain_text(path: str) -> str:
    with open(path, "rb") as fh:
        soup = BeautifulSoup(fh.read(), "lxml")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    return soup.get_text(" ", strip=True)


def chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    words  = text.split()
    chunks = []
    step   = max(1, size - overlap)
    for start in range(0, len(words), step):
        chunk = " ".join(words[start : start + size])
        if len(chunk.split()) >= 30:
            chunks.append(chunk)
    return chunks


# ------------------------------------------------------------------ index build
def build_faiss_index(tickers: list, force_rebuild: bool = False):
    if not force_rebuild and os.path.exists(FAISS_INDEX_PATH):
        print("FAISS index found on Drive. Loading...")
        index = faiss.read_index(FAISS_INDEX_PATH)
        with open(FAISS_CHUNKS_PATH, "rb") as fh:
            chunk_store = pickle.load(fh)
        print(f"  Loaded {index.ntotal} vectors, {len(chunk_store)} chunks.")
        return index, chunk_store

    print("Building FAISS index from SEC files on Drive...")
    all_chunks = []
    files      = os.listdir(DATA_DIR)

    for ticker in tickers:
        relevant = sorted(
            [f for f in files if f.startswith(ticker + "_")],
            reverse=True
        )
        for selected in relevant[:2]:
            path = os.path.join(DATA_DIR, selected)
            try:
                text   = html_to_plain_text(path)
                chunks = chunk_text(text)
                for chunk in chunks:
                    all_chunks.append({
                        "ticker": ticker,
                        "file":   selected,
                        "text":   chunk,
                    })
                print(f"  {ticker}: {len(chunks)} chunks from {selected}")
            except Exception as exc:
                print(f"  {ticker}: failed — {exc}")

    print(f"\nTotal chunks to embed: {len(all_chunks)}")
    print("Generating embeddings locally (2 to 3 minutes on Colab CPU)...")

    texts      = [c["text"] for c in all_chunks]
    embeddings = EMBED_MODEL.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    dim   = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)

    faiss.write_index(index, FAISS_INDEX_PATH)
    with open(FAISS_CHUNKS_PATH, "wb") as fh:
        pickle.dump(all_chunks, fh)

    print(f"\nFAISS index saved to Drive.")
    print(f"  Vectors indexed : {index.ntotal}")
    print(f"  Chunks stored   : {len(all_chunks)}")
    return index, all_chunks


faiss_index, chunk_store = build_faiss_index(["AAPL", "MSFT", "AMZN", "NVDA"])


# ------------------------------------------------------------------ retrieval
def vanilla_rag_retrieve(query: str, top_k: int = FAISS_K) -> list:
    query_vec          = EMBED_MODEL.encode([query], convert_to_numpy=True).astype(np.float32)
    distances, indices = faiss_index.search(query_vec, top_k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx < len(chunk_store):
            chunk          = chunk_store[idx].copy()
            chunk["score"] = float(dist)
            results.append(chunk)
    return results


def vanilla_rag_answer(query: str) -> str:
    chunks  = vanilla_rag_retrieve(query)
    context = "\n\n---\n\n".join([c["text"] for c in chunks])
    prompt  = (
        f"You are a financial analyst assistant. Answer the question using only "
        f"the context provided. Be precise with numbers.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\nAnswer:"
    )
    return llm_chat(prompt)


# Smoke test
print("\nRunning vanilla RAG smoke test...")
test_answer = vanilla_rag_answer("What was Apple's total revenue in fiscal year 2024?")
print(f"Answer: {test_answer}")

## BLOCK 7: PageIndex Retrieval (M2, adapted for Colab + OpenAI via CreateAI)

In [ ]:
# =============================================================================
# BLOCK 7: PageIndex Retrieval (M2, adapted for Colab + OpenAI via CreateAI)
# This is System 2 in the three-way evaluation.
# =============================================================================

!pip -q install pageindex

import json as _json
import warnings
from bs4 import XMLParsedAsHTMLWarning
from pageindex import PageIndexClient

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

pi_client = PageIndexClient(api_key="cfa65848f70c49f8ab9a0348d2546419")


# ------------------------------------------------------------------ tree utils
def flatten_tree(nodes: list, result: list = None) -> list:
    if result is None:
        result = []
    for node in nodes:
        result.append(node)
        if node.get("nodes"):
            flatten_tree(node["nodes"], result)
    return result


def tree_to_compact_str(nodes: list) -> str:
    flat = flatten_tree(nodes)
    return "\n".join([f"[{n['node_id']}] {n['title']}" for n in flat])


def get_nodes_by_ids(tree_nodes: list, target_ids: list) -> list:
    flat = flatten_tree(tree_nodes)
    return [n for n in flat if n["node_id"] in target_ids]


# ------------------------------------------------------------------ LLM tree search
def llm_tree_search(query: str, tree_nodes: list) -> list:
    """
    Asks the LLM to select the most relevant node IDs from the document tree.
    Falls back gracefully if the response cannot be parsed.
    """
    tree_str = tree_to_compact_str(tree_nodes[:50])
    prompt   = (
        f"You are a document navigation assistant.\n\n"
        f"Question: {query}\n\n"
        f"Below are section headings from a financial filing. "
        f"Return the IDs of the 2-3 sections most likely to contain the answer.\n\n"
        f"Sections:\n{tree_str}\n\n"
        f'Respond ONLY with valid JSON in this exact format: {{"node_list": ["id1","id2"]}}'
    )
    try:
        raw      = llm_chat(prompt, temperature=0.0)
        clean    = raw.strip().strip("```json").strip("```").strip()
        node_ids = _json.loads(clean).get("node_list", [])
        return node_ids
    except Exception:
        return []


# ------------------------------------------------------------------ scored fallback
SKIP_SECTIONS = {
    "risk", "7a", "competition", "forward-looking", "legal",
    "regulatory", "available information", "documents incorporated",
    "exhibit index"
}
FINANCE_TERMS = {
    "revenue", "net income", "financial", "statement",
    "operations", "results", "earnings"
}

def scored_fallback(flat_nodes: list) -> list:
    scored = []
    for node in flat_nodes:
        title = node.get("title", "").lower()
        text  = node.get("text",  "").lower()

        if any(bad in title for bad in SKIP_SECTIONS):
            continue

        score = 0
        if "item 8" in title:   score += 100
        elif "item 7" in title: score += 90

        if any(k in title or k in text for k in FINANCE_TERMS):
            score += 20
        if any(sym in text for sym in ["$", "million", "billion"]):
            score += 10

        if score > 0:
            scored.append((score, node))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [n for _, n in scored[:3]]


# ------------------------------------------------------------------ KG candidate lookup
def get_kg_candidates(query: str, tickers: list = None) -> list:
    if tickers is None:
        tickers = list({k.split("_")[0] for k in doc_map})

    candidates = []
    for ticker in tickers:
        filings = store.query_filings_for_ticker(ticker)
        for row in filings:
            acc     = row["accession_no"]
            map_key = f"{ticker}_{acc}"
            if map_key in doc_map:
                candidates.append({"ticker": ticker, "accession": acc})

    return candidates


# ------------------------------------------------------------------ main retrieval
def pageindex_retrieve(query: str, kg_candidates: list, top_docs: int = 1) -> list:
    retrieved = []

    for candidate in kg_candidates[:top_docs]:
        ticker    = candidate["ticker"]
        accession = candidate["accession"]
        map_key   = f"{ticker}_{accession}"

        if map_key not in doc_map:
            continue

        doc_id     = doc_map[map_key]
        tree_nodes = pi_client.get_tree(doc_id).get("result", [])
        flat_nodes = flatten_tree(tree_nodes)

        # Primary: LLM-guided node selection
        node_ids = llm_tree_search(query, tree_nodes)
        selected = get_nodes_by_ids(tree_nodes, node_ids)

        # Fallback: score-based section selection
        if not selected:
            selected = scored_fallback(flat_nodes)

        # Hard fallback: Item 7 / Item 8
        if not selected:
            selected = [
                n for n in flat_nodes
                if "item 7" in n.get("title", "").lower()
                or "item 8" in n.get("title", "").lower()
            ][:3]

        # Final fallback: first 3 nodes
        if not selected:
            selected = flat_nodes[:3]

        for node in selected:
            retrieved.append({
                "ticker":  ticker,
                "node_id": node["node_id"],
                "title":   node["title"],
                "text":    node.get("text", ""),
            })

    return retrieved


def pageindex_answer(query: str) -> str:
    candidates = get_kg_candidates(query)
    chunks     = pageindex_retrieve(query, candidates, top_docs=1)
    context    = "\n\n---\n\n".join(
        [f"[{c['ticker']} | {c['title']}]\n{c['text']}" for c in chunks]
    )
    prompt = (
        f"You are a financial analyst assistant. Answer the question using only "
        f"the context provided. Be precise with numbers.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\nAnswer:"
    )
    return llm_chat(prompt)


# Smoke test
print("Running PageIndex smoke test...")
test_pi = pageindex_answer("What was Apple's total revenue in fiscal year 2024?")
print(f"Answer: {test_pi}")

In [ ]:
# =============================================================================
# BLOCK 7 FIX: Ticker-aware candidate selection
# Detects which company the query is about and filters candidates accordingly.
# =============================================================================

QUERY_TICKER_MAP = {
    "apple":     "AAPL",
    "aapl":      "AAPL",
    "microsoft": "MSFT",
    "msft":      "MSFT",
    "amazon":    "AMZN",
    "amzn":      "AMZN",
    "nvidia":    "NVDA",
    "nvda":      "NVDA",
}

def detect_ticker_from_query(query: str) -> str:
    """Returns the ticker mentioned in the query, or None if not found."""
    lower = query.lower()
    for alias, ticker in QUERY_TICKER_MAP.items():
        if alias in lower:
            return ticker
    return None


def get_kg_candidates(query: str, tickers: list = None) -> list:
    """
    Returns candidates filtered to the company mentioned in the query.
    Falls back to all doc_map entries if no company is detected.
    """
    # Detect ticker from query text first
    detected = detect_ticker_from_query(query)

    if tickers is None:
        if detected:
            tickers = [detected]
        else:
            tickers = list({k.split("_")[0] for k in doc_map})

    candidates = []
    for ticker in tickers:
        filings = store.query_filings_for_ticker(ticker)
        for row in filings:
            acc     = row["accession_no"]
            map_key = f"{ticker}_{acc}"
            if map_key in doc_map:
                candidates.append({"ticker": ticker, "accession": acc})

    return candidates


# Re-run smoke test with the fix applied
print("Re-running PageIndex smoke test with ticker detection fix...")
test_pi = pageindex_answer("What was Apple's total revenue in fiscal year 2024?")
print(f"Answer: {test_pi}")

# SECTION 4: M3 Consistency Checker (Blocks 8-10)

## BLOCK 8: M3 — Claim Extractor

In [ ]:
# =============================================================================
# BLOCK 8: M3 — Claim Extractor
# Extracts verifiable numerical financial claims from LLM-generated answers.
# A claim is a (metric, ticker, year, value) tuple.
# =============================================================================

import re
import spacy

nlp_sm = spacy.load("en_core_web_sm")

METRIC_ALIASES = {
    "revenue":            "revenue",
    "net revenue":        "revenue",
    "total revenue":      "revenue",
    "sales":              "revenue",
    "net income":         "net_income",
    "net earnings":       "net_income",
    "net profit":         "net_income",
    "gross profit":       "gross_profit",
    "operating income":   "operating_income",
    "operating profit":   "operating_income",
    "earnings per share": "eps_basic",
    "eps":                "eps_basic",
    "basic eps":          "eps_basic",
}

TICKER_ALIASES = {
    "apple":     "AAPL",
    "aapl":      "AAPL",
    "microsoft": "MSFT",
    "msft":      "MSFT",
    "amazon":    "AMZN",
    "amzn":      "AMZN",
    "nvidia":    "NVDA",
    "nvda":      "NVDA",
    "google":    "GOOGL",
    "alphabet":  "GOOGL",
}

RE_DOLLAR_TRILLION = re.compile(r"\$\s*([\d,.]+)\s*trillion", re.IGNORECASE)
RE_DOLLAR_BILLION  = re.compile(r"\$\s*([\d,.]+)\s*billion",  re.IGNORECASE)
RE_DOLLAR_MILLION  = re.compile(r"\$\s*([\d,.]+)\s*million",  re.IGNORECASE)
RE_PLAIN_NUMBER    = re.compile(r"\$\s*([\d,.]+)")
RE_YEAR            = re.compile(r"\b(20\d{2})\b")


def parse_dollar_value(text: str):
    """Returns value in raw USD or None if no dollar amount found."""
    m = RE_DOLLAR_TRILLION.search(text)
    if m:
        return float(m.group(1).replace(",", "")) * 1e12
    m = RE_DOLLAR_BILLION.search(text)
    if m:
        return float(m.group(1).replace(",", "")) * 1e9
    m = RE_DOLLAR_MILLION.search(text)
    if m:
        return float(m.group(1).replace(",", "")) * 1e6
    m = RE_PLAIN_NUMBER.search(text)
    if m:
        return float(m.group(1).replace(",", ""))
    return None


def detect_ticker_from_text(text: str, context_ticker: str = None) -> str:
    lower = text.lower()
    for alias, ticker in TICKER_ALIASES.items():
        if alias in lower:
            return ticker
    return context_ticker


def detect_year_from_text(text: str, default_year: int = None) -> int:
    matches = RE_YEAR.findall(text)
    if matches:
        return int(matches[-1])
    return default_year


def detect_metric_from_sentence(sentence: str) -> str:
    lower = sentence.lower()
    for alias, metric_key in METRIC_ALIASES.items():
        if alias in lower:
            return metric_key
    return None


def extract_claims(answer_text: str, context_ticker: str = None, context_year: int = None) -> list:
    """
    Parses an LLM answer and returns a list of claim dicts:
      {metric, ticker, year, value, sentence}
    """
    doc    = nlp_sm(answer_text)
    claims = []

    for sent in doc.sents:
        sentence = sent.text.strip()
        value    = parse_dollar_value(sentence)
        if value is None:
            continue

        metric = detect_metric_from_sentence(sentence)
        if metric is None:
            continue

        ticker = detect_ticker_from_text(sentence, context_ticker)
        year   = detect_year_from_text(sentence, context_year)

        if ticker and year:
            claims.append({
                "metric":   metric,
                "ticker":   ticker,
                "year":     year,
                "value":    value,
                "sentence": sentence,
            })

    return claims


# Unit test
sample_answer = (
    "Apple reported total revenue of $391.0 billion in fiscal year 2024. "
    "Net income reached $93.7 billion for the same period."
)
claims = extract_claims(sample_answer, context_ticker="AAPL", context_year=2024)
print(f"Extracted {len(claims)} claims:")
for c in claims:
    print(f"  {c['ticker']} | {c['metric']} | {c['year']} | ${c['value']:,.0f}")

## BLOCK 9: M3 — KG Verifier

In [ ]:
# =============================================================================
# BLOCK 9: M3 — KG Verifier
# Cross-checks each extracted claim against FinancialMetric nodes in Neo4j.
# Returns verdict: VERIFIED, HALLUCINATED, or UNVERIFIABLE.
# Tolerance is 5% to account for rounding differences between sources.
# =============================================================================

TOLERANCE = 0.05

def verify_claim(claim: dict) -> dict:
    """
    Queries Neo4j for the ground truth value of the claimed metric.
    Augments the claim dict with verdict, kg_value, and error_pct.
    """
    result = store.query_metric(
        ticker = claim["ticker"],
        metric = claim["metric"],
        year   = claim["year"],
    )

    if result is None:
        return {
            **claim,
            "verdict":   "UNVERIFIABLE",
            "kg_value":  None,
            "error_pct": None,
        }

    kg_value  = result["value"]
    error_pct = abs(claim["value"] - kg_value) / abs(kg_value) if kg_value != 0 else None
    verdict   = "VERIFIED" if error_pct is not None and error_pct <= TOLERANCE else "HALLUCINATED"

    return {
        **claim,
        "verdict":   verdict,
        "kg_value":  kg_value,
        "error_pct": error_pct,
    }


def verify_all_claims(claims: list) -> list:
    return [verify_claim(c) for c in claims]


# Unit test using claims from Block 8
verified = verify_all_claims(claims)
print(f"Verified {len(verified)} claims:\n")
for v in verified:
    err_str = f"{v['error_pct']*100:.2f}%" if v["error_pct"] is not None else "N/A"
    kg_str  = f"${v['kg_value']:,.0f}" if v["kg_value"] is not None else "not in KG"
    print(f"  Verdict    : {v['verdict']}")
    print(f"  Metric     : {v['ticker']} {v['metric']} {v['year']}")
    print(f"  Claimed    : ${v['value']:,.0f}")
    print(f"  KG value   : {kg_str}")
    print(f"  Error      : {err_str}")
    print()

## BLOCK 10: M3 — Consistency Checker (Orchestrator)

In [ ]:
# =============================================================================
# BLOCK 10: M3 — Consistency Checker (Orchestrator)
# This is the core M3 deliverable. Full pipeline:
#   1. Retrieve context via PageIndex
#   2. Generate answer with LLM
#   3. Extract numerical claims from the answer
#   4. Verify each claim against Neo4j ground truth
#   5. Correct hallucinated values in the final answer
#   6. Return structured result with hallucination and verification rates
# =============================================================================

def build_corrected_answer(original_answer: str, verified_claims: list) -> str:
    """
    Rewrites sentences containing hallucinated values using the KG-verified figure.
    Leaves VERIFIED and UNVERIFIABLE sentences untouched.
    """
    corrected = original_answer
    for claim in verified_claims:
        if claim["verdict"] != "HALLUCINATED" or claim["kg_value"] is None:
            continue

        kg_billions = claim["kg_value"] / 1e9
        old_sentence = claim["sentence"]

        rewrite_prompt = (
            f"Rewrite the following sentence, replacing the incorrect financial "
            f"figure with the verified value of ${kg_billions:,.2f} billion. "
            f"Return only the corrected sentence, nothing else.\n\n"
            f'Sentence: "{old_sentence}"'
        )
        corrected_sentence = llm_chat(rewrite_prompt, temperature=0.0)
        corrected = corrected.replace(old_sentence, corrected_sentence)

    return corrected


def consistency_check(
    query:          str,
    context_ticker: str = None,
    context_year:   int = None,
    correct_output: bool = True,
) -> dict:
    """
    Full M3 pipeline for a single query.

    Returns
    -------
    {
        query               : original question
        raw_answer          : LLM answer before checking
        final_answer        : answer after correction
        claims              : extracted claim dicts
        verified_claims     : verified claim dicts
        hallucination_rate  : fraction of checkable claims that are wrong
        verification_rate   : fraction of claims that could be checked
        verdict_summary     : counts of VERIFIED / HALLUCINATED / UNVERIFIABLE
    }
    """
    # Step 1: retrieve and generate
    candidates = get_kg_candidates(query)
    chunks     = pageindex_retrieve(query, candidates, top_docs=1)

    if chunks and context_ticker is None:
        context_ticker = chunks[0]["ticker"]

    context = "\n\n---\n\n".join(
        [f"[{c['ticker']} | {c['title']}]\n{c['text']}" for c in chunks]
    )
    prompt = (
        f"You are a financial analyst assistant. Answer the question using only "
        f"the context provided. Be precise with numbers.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\nAnswer:"
    )
    raw_answer = llm_chat(prompt)

    # Step 2: extract and verify claims
    claims          = extract_claims(raw_answer, context_ticker, context_year)
    verified_claims = verify_all_claims(claims)

    # Step 3: compute metrics
    checkable  = [c for c in verified_claims if c["verdict"] != "UNVERIFIABLE"]
    halluc     = [c for c in checkable       if c["verdict"] == "HALLUCINATED"]

    hallucination_rate = len(halluc)    / len(checkable)       if checkable       else 0.0
    verification_rate  = len(checkable) / len(verified_claims) if verified_claims else 0.0

    verdicts = {"VERIFIED": 0, "HALLUCINATED": 0, "UNVERIFIABLE": 0}
    for c in verified_claims:
        verdicts[c["verdict"]] += 1

    # Step 4: correct hallucinated values
    final_answer = (
        build_corrected_answer(raw_answer, verified_claims)
        if correct_output else raw_answer
    )

    return {
        "query":              query,
        "raw_answer":         raw_answer,
        "final_answer":       final_answer,
        "claims":             claims,
        "verified_claims":    verified_claims,
        "hallucination_rate": hallucination_rate,
        "verification_rate":  verification_rate,
        "verdict_summary":    verdicts,
    }


# Smoke test
print("Running consistency checker smoke test...")
result = consistency_check(
    query          = "What was Apple's total revenue and net income in fiscal year 2024?",
    context_ticker = "AAPL",
    context_year   = 2024,
)

print(f"Raw answer    : {result['raw_answer']}")
print(f"Final answer  : {result['final_answer']}")
print(f"Claims found  : {len(result['verified_claims'])}")
print(f"Verdict       : {result['verdict_summary']}")
print(f"Hallucination : {result['hallucination_rate']*100:.1f}%")
print(f"Verification  : {result['verification_rate']*100:.1f}%")

In [ ]:
# =============================================================================
# BLOCK 10 FIX: Extract multiple claims from a single sentence
# The original extractor stopped at the first dollar amount per sentence.
# This version finds all dollar amounts within the same sentence.
# =============================================================================

import re

RE_ALL_AMOUNTS = re.compile(
    r"([\w\s,]+?)\s+(?:was|were|of|at|reached|totaled)?\s*"
    r"(\$[\d,.]+\s*(?:trillion|billion|million)?)",
    re.IGNORECASE
)

def extract_claims(answer_text: str, context_ticker: str = None, context_year: int = None) -> list:
    """
    Improved claim extractor. Finds all dollar amounts per sentence
    and attempts to match each to a metric keyword nearby.
    """
    doc    = nlp_sm(answer_text)
    claims = []
    seen   = set()

    for sent in doc.sents:
        sentence = sent.text.strip()
        year     = detect_year_from_text(sentence, context_year)
        ticker   = detect_ticker_from_text(sentence, context_ticker)

        if not ticker or not year:
            continue

        # Find all dollar amounts in the sentence
        dollar_positions = []
        for pattern in [RE_DOLLAR_TRILLION, RE_DOLLAR_BILLION, RE_DOLLAR_MILLION, RE_PLAIN_NUMBER]:
            for m in pattern.finditer(sentence):
                dollar_positions.append((m.start(), m.end(), parse_dollar_value(m.group(0))))

        # Remove duplicates — keep the most specific match at each position
        dollar_positions.sort(key=lambda x: x[0])
        filtered = []
        for start, end, val in dollar_positions:
            if val is None:
                continue
            if filtered and start < filtered[-1][1]:
                continue
            filtered.append((start, end, val))

        for start, end, value in filtered:
            # Look at the surrounding words (up to 6 words before the amount)
            prefix = sentence[:start].lower()
            words  = prefix.split()
            window = " ".join(words[-6:])

            metric = detect_metric_from_sentence(window + " " + sentence.lower())
            if metric is None:
                continue

            dedup_key = (ticker, metric, year)
            if dedup_key in seen:
                continue
            seen.add(dedup_key)

            claims.append({
                "metric":   metric,
                "ticker":   ticker,
                "year":     year,
                "value":    value,
                "sentence": sentence,
            })

    return claims


# Verify the fix extracts both claims from a single sentence
sample_answer = (
    "Apple's total revenue in fiscal year 2024 was $391,035 million, "
    "and the net income was $93,736 million."
)
test_claims = extract_claims(sample_answer, context_ticker="AAPL", context_year=2024)
print(f"Extracted {len(test_claims)} claims:")
for c in test_claims:
    print(f"  {c['ticker']} | {c['metric']} | {c['year']} | ${c['value']:,.0f}")

In [ ]:
# =============================================================================
# BLOCK 10 FIX v2: Metric-first claim extraction
# Scans for metric keywords first, then finds the nearest dollar amount
# after each keyword. More reliable than scanning dollar amounts backwards.
# =============================================================================

def extract_claims(answer_text: str, context_ticker: str = None, context_year: int = None) -> list:
    """
    For each sentence:
      1. Find all metric keywords and their positions
      2. For each keyword, find the nearest dollar amount that follows it
      3. Build a claim from (metric, ticker, year, value)
    """
    doc    = nlp_sm(answer_text)
    claims = []
    seen   = set()

    for sent in doc.sents:
        sentence = sent.text.strip()
        lower    = sentence.lower()
        year     = detect_year_from_text(sentence, context_year)
        ticker   = detect_ticker_from_text(sentence, context_ticker)

        if not ticker or not year:
            continue

        # Find all dollar amounts and their start positions in the sentence
        dollar_hits = []
        for pattern in [RE_DOLLAR_TRILLION, RE_DOLLAR_BILLION, RE_DOLLAR_MILLION]:
            for m in pattern.finditer(sentence):
                val = parse_dollar_value(m.group(0))
                if val is not None:
                    dollar_hits.append((m.start(), val))

        if not dollar_hits:
            continue

        dollar_hits.sort(key=lambda x: x[0])

        # Find all metric keyword positions in the sentence
        metric_hits = []
        for alias, metric_key in METRIC_ALIASES.items():
            idx = lower.find(alias)
            while idx != -1:
                metric_hits.append((idx, metric_key, alias))
                idx = lower.find(alias, idx + 1)

        if not metric_hits:
            continue

        metric_hits.sort(key=lambda x: x[0])

        # For each metric keyword, find the nearest dollar amount after it
        for m_pos, metric_key, alias in metric_hits:
            # Find dollar amounts that come after this metric keyword
            candidates = [(d_pos, val) for d_pos, val in dollar_hits if d_pos > m_pos]
            if not candidates:
                # Allow dollar amounts slightly before (within 30 chars) e.g. "was $X in net income"
                candidates = [(d_pos, val) for d_pos, val in dollar_hits if d_pos >= m_pos - 30]
            if not candidates:
                continue

            # Take the closest dollar amount
            nearest_pos, value = min(candidates, key=lambda x: abs(x[0] - m_pos))

            dedup_key = (ticker, metric_key, year)
            if dedup_key in seen:
                continue
            seen.add(dedup_key)

            claims.append({
                "metric":   metric_key,
                "ticker":   ticker,
                "year":     year,
                "value":    value,
                "sentence": sentence,
            })

    return claims


# Test on single sentence case
sample_single = (
    "Apple's total revenue in fiscal year 2024 was $391,035 million, "
    "and the net income was $93,736 million."
)
test1 = extract_claims(sample_single, context_ticker="AAPL", context_year=2024)
print(f"Single sentence test — extracted {len(test1)} claims:")
for c in test1:
    print(f"  {c['ticker']} | {c['metric']} | {c['year']} | ${c['value']:,.0f}")

# Test on multi sentence case
sample_multi = (
    "Apple reported total revenue of $391.0 billion in fiscal year 2024. "
    "Net income reached $93.7 billion for the same period. "
    "Gross profit was $170.0 billion."
)
test2 = extract_claims(sample_multi, context_ticker="AAPL", context_year=2024)
print(f"\nMulti sentence test — extracted {len(test2)} claims:")
for c in test2:
    print(f"  {c['ticker']} | {c['metric']} | {c['year']} | ${c['value']:,.0f}")

In [ ]:
# =============================================================================
# BLOCK 10 EDGE CASE TESTS
# Tests all realistic answer formats the LLM might produce.
# Fix any failures before running the full evaluation.
# =============================================================================

test_cases = [
    # Case 1: Billions with decimal
    {
        "name": "Billions with decimal",
        "text": "Apple's revenue was $391.0 billion in fiscal year 2024.",
        "ticker": "AAPL", "year": 2024,
        "expected": [("revenue", 391e9)]
    },
    # Case 2: Millions without year in sentence (relies on context_year)
    {
        "name": "Millions, year from context",
        "text": "Net income was $93,736 million.",
        "ticker": "AAPL", "year": 2024,
        "expected": [("net_income", 93736e6)]
    },
    # Case 3: Multiple metrics in one sentence
    {
        "name": "Multiple metrics one sentence",
        "text": "Revenue was $391,035 million and net income was $93,736 million in fiscal 2024.",
        "ticker": "AAPL", "year": 2024,
        "expected": [("revenue", 391035e6), ("net_income", 93736e6)]
    },
    # Case 4: Company name in sentence (no context_ticker)
    {
        "name": "Ticker from sentence text",
        "text": "Microsoft reported total revenue of $245,122 million in fiscal year 2024.",
        "ticker": None, "year": None,
        "expected": [("revenue", 245122e6)]
    },
    # Case 5: Earnings per share (small number, no billion/million suffix)
    {
        "name": "EPS — small number",
        "text": "Apple's basic EPS was $6.11 in fiscal year 2024.",
        "ticker": "AAPL", "year": 2024,
        "expected": [("eps_basic", 6.11)]
    },
    # Case 6: Gross profit and operating income together
    {
        "name": "Gross profit and operating income",
        "text": "Gross profit reached $170.0 billion and operating income was $123.0 billion in 2024.",
        "ticker": "AAPL", "year": 2024,
        "expected": [("gross_profit", 170e9), ("operating_income", 123e9)]
    },
    # Case 7: Phrasing with "totaled"
    {
        "name": "Totaled phrasing",
        "text": "Net revenue totaled $391.0 billion for the fiscal year 2024.",
        "ticker": "AAPL", "year": 2024,
        "expected": [("revenue", 391e9)]
    },
    # Case 8: No dollar amount — should return empty
    {
        "name": "No dollar amount",
        "text": "Apple had strong performance in fiscal year 2024.",
        "ticker": "AAPL", "year": 2024,
        "expected": []
    },
    # Case 9: Multiple companies in one answer
    {
        "name": "Multiple companies",
        "text": "Apple revenue was $391.0 billion in 2024. Microsoft revenue was $245.1 billion in 2024.",
        "ticker": None, "year": None,
        "expected": [("revenue", 391e9), ("revenue", 245.1e9)]
    },
    # Case 10: NVDA with fiscal year phrasing
    {
        "name": "NVIDIA operating income",
        "text": "NVIDIA's operating income was $134.0 billion in fiscal year 2025.",
        "ticker": None, "year": None,
        "expected": [("operating_income", 134e9)]
    },
]

# Run all tests
print("Running edge case tests...\n")
passed = 0
failed = 0

for tc in test_cases:
    claims = extract_claims(tc["text"], context_ticker=tc["ticker"], context_year=tc["year"])
    expected_count = len(tc["expected"])
    actual_count   = len(claims)

    status = "PASS" if actual_count == expected_count else "FAIL"
    if status == "PASS":
        passed += 1
    else:
        failed += 1

    print(f"[{status}] {tc['name']}")
    print(f"       Expected {expected_count} claims, got {actual_count}")

    if actual_count > 0:
        for c in claims:
            print(f"       {c['metric']} | ${c['value']:,.0f}")

    if status == "FAIL":
        print(f"       Expected: {tc['expected']}")
    print()

print(f"Results: {passed} passed, {failed} failed out of {len(test_cases)} tests")

In [ ]:
# =============================================================================
# BLOCK 10 FIX v3: Add EPS-specific extraction
# EPS values are small numbers like $6.11 — no billion/million suffix.
# Requires a dedicated pattern that only triggers near EPS keywords.
# =============================================================================

RE_EPS_AMOUNT = re.compile(r"\$\s*([\d]+\.[\d]+|\d+)", re.IGNORECASE)

EPS_KEYWORDS = {"earnings per share", "eps", "basic eps", "diluted eps", "per share"}

def extract_claims(answer_text: str, context_ticker: str = None, context_year: int = None) -> list:
    doc    = nlp_sm(answer_text)
    claims = []
    seen   = set()

    for sent in doc.sents:
        sentence = sent.text.strip()
        lower    = sentence.lower()
        year     = detect_year_from_text(sentence, context_year)
        ticker   = detect_ticker_from_text(sentence, context_ticker)

        if not ticker or not year:
            continue

        # ------------------------------------------------ EPS special case
        # Handle separately because EPS values are small decimals
        # and would be swamped by the general billion/million patterns
        is_eps_sentence = any(kw in lower for kw in EPS_KEYWORDS)
        if is_eps_sentence:
            for m in RE_EPS_AMOUNT.finditer(sentence):
                val = float(m.group(1).replace(",", ""))
                # EPS values are always small — ignore if looks like a year or large number
                if val > 1000:
                    continue
                dedup_key = ("eps_basic", ticker, year)
                if dedup_key in seen:
                    continue
                seen.add(dedup_key)
                claims.append({
                    "metric":   "eps_basic",
                    "ticker":   ticker,
                    "year":     year,
                    "value":    val,
                    "sentence": sentence,
                })

        # ------------------------------------------------ General case
        # Find all billion/million dollar amounts
        dollar_hits = []
        for pattern in [RE_DOLLAR_TRILLION, RE_DOLLAR_BILLION, RE_DOLLAR_MILLION]:
            for m in pattern.finditer(sentence):
                val = parse_dollar_value(m.group(0))
                if val is not None:
                    dollar_hits.append((m.start(), val))

        if not dollar_hits:
            continue

        dollar_hits.sort(key=lambda x: x[0])

        # Find all metric keyword positions
        metric_hits = []
        for alias, metric_key in METRIC_ALIASES.items():
            if metric_key == "eps_basic":
                continue    # already handled above
            idx = lower.find(alias)
            while idx != -1:
                metric_hits.append((idx, metric_key, alias))
                idx = lower.find(alias, idx + 1)

        if not metric_hits:
            continue

        metric_hits.sort(key=lambda x: x[0])

        for m_pos, metric_key, alias in metric_hits:
            candidates = [(d_pos, val) for d_pos, val in dollar_hits if d_pos > m_pos]
            if not candidates:
                candidates = [(d_pos, val) for d_pos, val in dollar_hits if d_pos >= m_pos - 30]
            if not candidates:
                continue

            nearest_pos, value = min(candidates, key=lambda x: abs(x[0] - m_pos))

            dedup_key = (ticker, metric_key, year)
            if dedup_key in seen:
                continue
            seen.add(dedup_key)

            claims.append({
                "metric":   metric_key,
                "ticker":   ticker,
                "year":     year,
                "value":    value,
                "sentence": sentence,
            })

    return claims


# Re-run all edge case tests
print("Re-running edge case tests...\n")
passed = 0
failed = 0

for tc in test_cases:
    claims = extract_claims(tc["text"], context_ticker=tc["ticker"], context_year=tc["year"])
    expected_count = len(tc["expected"])
    actual_count   = len(claims)

    status = "PASS" if actual_count == expected_count else "FAIL"
    if status == "PASS":
        passed += 1
    else:
        failed += 1

    print(f"[{status}] {tc['name']}")
    print(f"       Expected {expected_count} claims, got {actual_count}")
    if actual_count > 0:
        for c in claims:
            print(f"       {c['metric']} | ${c['value']:,.0f}")
    if status == "FAIL":
        print(f"       Expected: {tc['expected']}")
    print()

print(f"Results: {passed} passed, {failed} failed out of {len(test_cases)} tests")

# SECTION 5: Evaluation (Blocks 11-12)

## Block 11 Fix — FAISS index rebuild with date sort

In [ ]:
# =============================================================================
# BLOCK 11 FIX: Correct file selection for FAISS + expand MSFT doc_map
# Root cause: files were sorted by accession number, not filing date.
# Fix: extract date from filename and sort properly before selecting.
# =============================================================================

import re as _re

def extract_filing_date_from_filename(filename: str) -> str:
    """
    Extracts the filing date from the SEC filename.
    Filename format: TICKER_ACCESSION_primarydoc.htm
    Accession format: XXXXXXXXXX-YY-NNNNNN where YY is the year filed.
    Falls back to accession number sort if date not found.
    """
    # Try to extract date from the document name part (e.g. aapl-20240928)
    m = _re.search(r"(\d{8})\.", filename)
    if m:
        return m.group(1)
    # Fall back to accession year (e.g. -25- means filed in 2025)
    m = _re.search(r"-(\d{2})-\d{6}", filename)
    if m:
        return m.group(1)
    return filename


def build_faiss_index_fixed(tickers: list, force_rebuild: bool = True):
    """
    Rebuilds FAISS index with correct date-based file selection.
    Selects the 2 most recent filings per ticker by document date.
    """
    if not force_rebuild and os.path.exists(FAISS_INDEX_PATH):
        print("FAISS index found. Loading...")
        index = faiss.read_index(FAISS_INDEX_PATH)
        with open(FAISS_CHUNKS_PATH, "rb") as fh:
            chunk_store = pickle.load(fh)
        print(f"  Loaded {index.ntotal} vectors, {len(chunk_store)} chunks.")
        return index, chunk_store

    print("Rebuilding FAISS index with date-sorted file selection...")
    all_chunks = []
    files      = os.listdir(DATA_DIR)

    for ticker in tickers:
        relevant = [f for f in files if f.startswith(ticker + "_")]

        # Sort by document date descending — most recent first
        relevant.sort(key=extract_filing_date_from_filename, reverse=True)

        print(f"\n  {ticker} filing order after date sort:")
        for f in relevant[:4]:
            print(f"    {f}")

        for selected in relevant[:2]:
            path = os.path.join(DATA_DIR, selected)
            try:
                text   = html_to_plain_text(path)
                chunks = chunk_text(text)
                for chunk in chunks:
                    all_chunks.append({
                        "ticker": ticker,
                        "file":   selected,
                        "text":   chunk,
                    })
                print(f"  Selected: {selected} — {len(chunks)} chunks")
            except Exception as exc:
                print(f"  Failed: {selected} — {exc}")

    print(f"\nTotal chunks to embed: {len(all_chunks)}")
    print("Generating embeddings (1-2 minutes on T4)...")

    texts      = [c["text"] for c in all_chunks]
    embeddings = EMBED_MODEL.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    dim   = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)

    faiss.write_index(index, FAISS_INDEX_PATH)
    with open(FAISS_CHUNKS_PATH, "wb") as fh:
        pickle.dump(all_chunks, fh)

    print(f"\nFAISS index rebuilt and saved to Drive.")
    print(f"  Vectors indexed : {index.ntotal}")
    print(f"  Chunks stored   : {len(all_chunks)}")
    return index, all_chunks


# Expand doc_map with additional MSFT filings
# These accession numbers are already downloaded on your Drive
doc_map.update({
    "MSFT_0000950170-24-087843": None,   # placeholder — needs PageIndex upload
    "AAPL_0000320193-24-000123": None,   # already in doc_map but confirm
})

# Keep only entries with valid PageIndex doc IDs
doc_map = {k: v for k, v in doc_map.items() if v is not None}
save_doc_map(DOC_MAP_PATH, doc_map)

print(f"Active doc_map entries: {len(doc_map)}")
for k, v in doc_map.items():
    print(f"  {k}: {v}")

# Rebuild FAISS with correct file selection
faiss_index, chunk_store = build_faiss_index_fixed(
    ["AAPL", "MSFT", "AMZN", "NVDA"],
    force_rebuild=True
)

# Verify correct files were picked
print("\nVerification — files in FAISS index:")
seen_files = list({c["file"] for c in chunk_store})
for f in sorted(seen_files):
    print(f"  {f}")

## BLOCK 11: Evaluation Loop (re-run with corrected FAISS index)

In [ ]:
# =============================================================================
# BLOCK 11: Evaluation Loop (re-run with corrected FAISS index)
# =============================================================================

import pandas as pd

EVAL_QUERIES = [
    {"query": "What was Apple's total revenue in fiscal year 2024?",         "ticker": "AAPL", "year": 2024},
    {"query": "What was Apple's net income in fiscal year 2024?",            "ticker": "AAPL", "year": 2024},
    {"query": "What was Apple's gross profit in fiscal year 2024?",          "ticker": "AAPL", "year": 2024},
    {"query": "What was Apple's operating income in fiscal year 2024?",      "ticker": "AAPL", "year": 2024},
    {"query": "What was Apple's total revenue in fiscal year 2023?",         "ticker": "AAPL", "year": 2023},
    {"query": "What was Apple's net income in fiscal year 2023?",            "ticker": "AAPL", "year": 2023},
    {"query": "What was Microsoft's total revenue in fiscal year 2024?",     "ticker": "MSFT", "year": 2024},
    {"query": "What was Microsoft's net income in fiscal year 2024?",        "ticker": "MSFT", "year": 2024},
    {"query": "What was Microsoft's gross profit in fiscal year 2024?",      "ticker": "MSFT", "year": 2024},
    {"query": "What was Microsoft's operating income in fiscal year 2024?",  "ticker": "MSFT", "year": 2024},
]


def eval_single(query: str, ticker: str, year: int, answer: str) -> dict:
    claims  = extract_claims(answer, context_ticker=ticker, context_year=year)
    vclaims = verify_all_claims(claims)

    checkable = [c for c in vclaims if c["verdict"] != "UNVERIFIABLE"]
    halluc    = [c for c in checkable if c["verdict"] == "HALLUCINATED"]

    return {
        "query":              query,
        "answer":             answer,
        "n_claims":           len(vclaims),
        "hallucination_rate": len(halluc)    / len(checkable) if checkable else 0.0,
        "verification_rate":  len(checkable) / len(vclaims)   if vclaims   else 0.0,
        "verdict_summary": {
            "VERIFIED":     sum(1 for c in vclaims if c["verdict"] == "VERIFIED"),
            "HALLUCINATED": sum(1 for c in vclaims if c["verdict"] == "HALLUCINATED"),
            "UNVERIFIABLE": sum(1 for c in vclaims if c["verdict"] == "UNVERIFIABLE"),
        }
    }


def run_vanilla_rag_eval(item: dict) -> dict:
    answer = vanilla_rag_answer(item["query"])
    result = eval_single(item["query"], item["ticker"], item["year"], answer)
    return {"system": "Vanilla RAG", **result}


def run_pageindex_eval(item: dict) -> dict:
    answer = pageindex_answer(item["query"])
    result = eval_single(item["query"], item["ticker"], item["year"], answer)
    return {"system": "PageIndex RAG", **result}


def run_reasonkg_eval(item: dict) -> dict:
    result = consistency_check(
        query          = item["query"],
        context_ticker = item["ticker"],
        context_year   = item["year"],
        correct_output = True,
    )
    return {
        "system":             "ReasonKG",
        "query":              item["query"],
        "answer":             result["final_answer"],
        "n_claims":           len(result["verified_claims"]),
        "hallucination_rate": result["hallucination_rate"],
        "verification_rate":  result["verification_rate"],
        "verdict_summary":    result["verdict_summary"],
    }


print(f"Running evaluation across {len(EVAL_QUERIES)} queries x 3 systems...\n")

all_results = []

for i, item in enumerate(EVAL_QUERIES):
    print(f"Query {i+1}/{len(EVAL_QUERIES)}: {item['query'][:65]}...")

    v_result  = run_vanilla_rag_eval(item)
    pi_result = run_pageindex_eval(item)
    rk_result = run_reasonkg_eval(item)

    all_results.extend([v_result, pi_result, rk_result])

    print(f"  Vanilla RAG   — hallucination: {v_result['hallucination_rate']*100:.0f}%  claims: {v_result['n_claims']}")
    print(f"  PageIndex RAG — hallucination: {pi_result['hallucination_rate']*100:.0f}%  claims: {pi_result['n_claims']}")
    print(f"  ReasonKG      — hallucination: {rk_result['hallucination_rate']*100:.0f}%  claims: {rk_result['n_claims']}")
    print()

    time.sleep(0.5)

# Summary
df_results = pd.DataFrame(all_results)

df_summary = (
    df_results
    .groupby("system")[["hallucination_rate", "verification_rate", "n_claims"]]
    .mean()
    .round(4)
    .reset_index()
)
df_summary["hallucination_pct"] = (df_summary["hallucination_rate"] * 100).round(1)
df_summary["verification_pct"]  = (df_summary["verification_rate"]  * 100).round(1)

order = ["Vanilla RAG", "PageIndex RAG", "ReasonKG"]
df_summary["system"] = pd.Categorical(df_summary["system"], categories=order, ordered=True)
df_summary = df_summary.sort_values("system").reset_index(drop=True)

print("\n=== Evaluation Summary ===")
print(df_summary[["system", "hallucination_pct", "verification_pct", "n_claims"]].to_string(index=False))

# Save to Drive
df_results.to_csv("/content/drive/MyDrive/eval_results_full.csv",    index=False)
df_summary.to_csv("/content/drive/MyDrive/eval_results_summary.csv", index=False)
print("\nResults saved to Google Drive.")

In [ ]:
# =============================================================================
# BLOCK 12: Results Visualization
# Produces 3 publication-ready plots saved to Google Drive.
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np

SYSTEM_ORDER  = ["Vanilla RAG", "PageIndex RAG", "ReasonKG"]
SYSTEM_COLORS = ["#C0392B", "#2980B9", "#27AE60"]
SAVE_DIR      = "/content/drive/MyDrive"

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         11,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
})


# ------------------------------------------------------------------ Figure 1
# Hallucination Rate by System (bar chart)
fig, ax = plt.subplots(figsize=(8, 5))

halluc_vals = [
    df_summary.loc[df_summary["system"] == s, "hallucination_pct"].values[0]
    for s in SYSTEM_ORDER
]

bars = ax.bar(SYSTEM_ORDER, halluc_vals, color=SYSTEM_COLORS, width=0.5, edgecolor="white", linewidth=1.2)

ax.set_ylabel("Hallucination Rate (%)", fontsize=12)
ax.set_title("Hallucination Rate by System\n(lower is better)", fontsize=13, fontweight="bold")
ax.set_ylim(0, max(halluc_vals) * 1.4 + 2)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))

for bar, val in zip(bars, halluc_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f"{val:.1f}%",
        ha="center", va="bottom", fontsize=11, fontweight="bold"
    )

ax.axhline(y=0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/fig1_hallucination_rate.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 1 saved.")


# ------------------------------------------------------------------ Figure 2
# Verification Coverage by System (bar chart)
fig, ax = plt.subplots(figsize=(8, 5))

verif_vals = [
    df_summary.loc[df_summary["system"] == s, "verification_pct"].values[0]
    for s in SYSTEM_ORDER
]

bars = ax.bar(SYSTEM_ORDER, verif_vals, color=SYSTEM_COLORS, width=0.5, edgecolor="white", linewidth=1.2)

ax.set_ylabel("Verification Coverage (%)", fontsize=12)
ax.set_title("KG Verification Coverage by System\n(higher is better)", fontsize=13, fontweight="bold")
ax.set_ylim(0, 120)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))

for bar, val in zip(bars, verif_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.8,
        f"{val:.1f}%",
        ha="center", va="bottom", fontsize=11, fontweight="bold"
    )

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/fig2_verification_rate.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 2 saved.")


# ------------------------------------------------------------------ Figure 3
# Per-query hallucination heatmap
df_pivot = df_results.pivot_table(
    index="query", columns="system", values="hallucination_rate"
)[SYSTEM_ORDER]

# Shorten query labels
short_labels = [
    q.replace("What was ", "").replace(" in fiscal year", " FY").replace("?", "")
    for q in df_pivot.index
]
df_pivot.index = short_labels

fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(
    df_pivot.values,
    aspect="auto",
    cmap="RdYlGn_r",
    vmin=0, vmax=1
)

ax.set_xticks(range(len(SYSTEM_ORDER)))
ax.set_xticklabels(SYSTEM_ORDER, fontsize=11, fontweight="bold")
ax.set_yticks(range(len(df_pivot.index)))
ax.set_yticklabels(df_pivot.index, fontsize=9)

for i in range(len(df_pivot.index)):
    for j in range(len(SYSTEM_ORDER)):
        val = df_pivot.values[i, j]
        label = "HALL." if val > 0 else "OK"
        color = "white" if val > 0.5 else "black"
        ax.text(j, i, label, ha="center", va="center", fontsize=9, color=color, fontweight="bold")

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Hallucination Rate (0 = none, 1 = hallucinated)", fontsize=10)
ax.set_title("Per-Query Hallucination Heatmap\n(green = correct, red = hallucinated)",
             fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/fig3_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 3 saved.")


# ------------------------------------------------------------------ Figure 4
# Side-by-side grouped bar: hallucination vs verification per system
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x     = np.arange(len(SYSTEM_ORDER))
width = 0.5

# Left: hallucination
axes[0].bar(x, halluc_vals, width, color=SYSTEM_COLORS, edgecolor="white")
axes[0].set_xticks(x)
axes[0].set_xticklabels(SYSTEM_ORDER, fontsize=10)
axes[0].set_ylabel("Rate (%)")
axes[0].set_title("Hallucination Rate\n(lower is better)", fontweight="bold")
axes[0].set_ylim(0, max(halluc_vals) * 1.5 + 5)
for xi, val in zip(x, halluc_vals):
    axes[0].text(xi, val + 0.3, f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

# Right: verification coverage
axes[1].bar(x, verif_vals, width, color=SYSTEM_COLORS, edgecolor="white")
axes[1].set_xticks(x)
axes[1].set_xticklabels(SYSTEM_ORDER, fontsize=10)
axes[1].set_ylabel("Rate (%)")
axes[1].set_title("Verification Coverage\n(higher is better)", fontweight="bold")
axes[1].set_ylim(0, 120)
for xi, val in zip(x, verif_vals):
    axes[1].text(xi, val + 0.8, f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

fig.suptitle("ReasonKG System Evaluation — CSE 579 Group 6",
             fontsize=14, fontweight="bold", y=1.02)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/fig4_combined_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 4 saved.")

print("\nAll figures saved to Google Drive.")
print("Path: /content/drive/MyDrive/")
print("Files: fig1_hallucination_rate.png, fig2_verification_rate.png,")
print("       fig3_heatmap.png, fig4_combined_summary.png")

# Section 6: Final Expansion — All Filings + Hard Evaluation

## STEP 1: Check PageIndex upload status for all 26 filings

In [ ]:
# =============================================================================
# STEP 1: Check PageIndex upload status for all 26 filings
# Run this first to see exactly what needs uploading
# =============================================================================

files_on_drive = sorted(os.listdir(DATA_DIR))

print(f"Total files on Drive: {len(files_on_drive)}")
print(f"Currently in PageIndex doc_map: {len(doc_map)}\n")

missing = []
already = []

for f in files_on_drive:
    parts     = f.split("_")
    ticker    = parts[0]
    accession = parts[1]
    map_key   = f"{ticker}_{accession}"

    if map_key in doc_map:
        already.append(map_key)
    else:
        missing.append((map_key, os.path.join(DATA_DIR, f)))

print(f"Already in PageIndex ({len(already)}):")
for k in already:
    print(f"  {k}")

print(f"\nNeeds uploading ({len(missing)}):")
for k, _ in missing:
    print(f"  {k}")

## STEP 2: Upload all 24 remaining filings to PageIndex

In [ ]:
# =============================================================================
# STEP 2: Upload all 24 remaining filings to PageIndex
# Uses correct method: pi_client.submit_document()
# =============================================================================

import time

def upload_to_pageindex(map_key: str, filepath: str) -> str:
    """
    Uploads a single filing to PageIndex using submit_document().
    Returns the document ID or None if failed.
    """
    try:
        with open(filepath, "rb") as fh:
            content = fh.read()

        filename = os.path.basename(filepath)

        # Check what parameters submit_document accepts
        import inspect
        sig = inspect.signature(pi_client.submit_document)

        result = pi_client.submit_document(
            content  = content,
            filename = filename,
        )

        # Try all possible key names for the document ID
        doc_id = (
            result.get("document_id")
            or result.get("id")
            or result.get("doc_id")
            or result.get("documentId")
            or result.get("document", {}).get("id")
        )

        if not doc_id and isinstance(result, dict):
            print(f"    Full response: {result}")

        return doc_id

    except Exception as exc:
        # Print full error and the signature so we can fix it
        import inspect
        try:
            sig = inspect.signature(pi_client.submit_document)
            print(f"    submit_document signature: {sig}")
        except:
            pass
        print(f"    Upload error: {exc}")
        return None


# Test with just the first file before running all 24
test_key, test_path = missing[0]
print(f"Testing upload with: {test_key}")
test_doc_id = upload_to_pageindex(test_key, test_path)
print(f"Result: {test_doc_id}")

In [ ]:
# =============================================================================
# STEP 2: Upload all 24 remaining filings to PageIndex
# submit_document() takes file_path as a string, not content bytes.
# =============================================================================

import time

def upload_to_pageindex(map_key: str, filepath: str) -> str:
    try:
        result = pi_client.submit_document(file_path=filepath)

        print(f"    Raw response: {result}")

        doc_id = (
            result.get("document_id")
            or result.get("id")
            or result.get("doc_id")
            or result.get("documentId")
            or result.get("document", {}).get("id")
        )

        return doc_id

    except Exception as exc:
        print(f"    Upload error: {exc}")
        return None


# Test with just the first file
test_key, test_path = missing[0]
print(f"Testing upload with: {test_key}")
print(f"File path: {test_path}")
test_doc_id = upload_to_pageindex(test_key, test_path)
print(f"Extracted doc_id: {test_doc_id}")

In [ ]:
# =============================================================================
# STEP 2 FIX v2: Corrected PDF margins and cell configuration
# =============================================================================

from fpdf import FPDF
import os

def html_to_pdf(html_path: str, pdf_path: str) -> bool:
    try:
        # Extract clean text
        with open(html_path, "rb") as fh:
            soup = BeautifulSoup(fh.read(), "lxml")
        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()
        text = soup.get_text(" ", strip=True)
        text = text[:500000]

        # Build PDF with explicit wide margins
        pdf = FPDF(orientation="P", unit="mm", format="A4")
        pdf.set_margins(left=10, top=10, right=10)
        pdf.set_auto_page_break(auto=True, margin=10)
        pdf.add_page()
        pdf.set_font("Helvetica", size=8)

        # Page width minus margins = usable width
        usable_width = pdf.w - 20

        # Write in small chunks with explicit width
        chunk_size = 500
        for i in range(0, len(text), chunk_size):
            chunk = text[i : i + chunk_size]
            chunk = chunk.encode("latin-1", errors="replace").decode("latin-1")
            pdf.multi_cell(w=usable_width, h=4, txt=chunk)

        pdf.output(pdf_path)
        size_mb = os.path.getsize(pdf_path) / 1e6
        print(f"    PDF created: {size_mb:.1f} MB")
        return True

    except Exception as exc:
        print(f"    PDF creation failed: {exc}")
        return False


# Re-test
test_key, test_path = missing[0]
print(f"Testing with: {test_key}\n")
test_doc_id = upload_pdf_to_pageindex(test_key, test_path)
print(f"\nDoc ID: {test_doc_id}")

## STEP 3: Hybrid Retrieval — PageIndex where available, KG-FAISS elsewhere

In [ ]:
# =============================================================================
# STEP 3: Hybrid Retrieval — PageIndex where available, KG-FAISS elsewhere
#
# Logic:
#   1. Detect ticker from query
#   2. Check if that ticker+accession exists in doc_map (PageIndex)
#      YES -> use PageIndex tree-based retrieval
#      NO  -> use KG-guided FAISS retrieval
#   3. M3 consistency checker works identically on both paths
#
# KG-guided FAISS differs from Vanilla FAISS in one key way:
#   Vanilla FAISS: embeds query, retrieves top-k chunks globally
#   KG-FAISS:      first queries Neo4j for entities related to the ticker,
#                  then filters FAISS results to chunks from that ticker only
# =============================================================================

# ------------------------------------------------------------------ KG-guided FAISS
def kg_guided_retrieve(query: str, ticker: str, top_k: int = 3) -> list:
    """
    Retrieves chunks from FAISS filtered to a specific ticker.
    Uses Neo4j to confirm the ticker has data before retrieval.
    This is the KG-grounding step — retrieval is anchored to a
    verified KG entity rather than relying on embedding similarity alone.
    """
    # Step 1: confirm ticker exists in KG
    kg_check = store.run("""
        MATCH (c:Company {ticker: $ticker})
        RETURN c.ticker AS ticker
    """, ticker=ticker)

    if not kg_check:
        print(f"  Ticker {ticker} not found in KG — falling back to global FAISS")
        return vanilla_rag_retrieve(query, top_k=top_k)

    # Step 2: embed query
    query_vec = EMBED_MODEL.encode([query], convert_to_numpy=True).astype(np.float32)

    # Step 3: retrieve more candidates than needed, then filter by ticker
    distances, indices = faiss_index.search(query_vec, top_k * 10)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx >= len(chunk_store):
            continue
        chunk = chunk_store[idx]
        if chunk["ticker"] == ticker:
            results.append({**chunk, "score": float(dist)})
        if len(results) >= top_k:
            break

    # If not enough ticker-specific chunks found, relax to global
    if not results:
        print(f"  No KG-filtered chunks found for {ticker} — using global FAISS")
        return vanilla_rag_retrieve(query, top_k=top_k)

    return results


# ------------------------------------------------------------------ hybrid retrieval
def hybrid_retrieve(query: str, ticker: str) -> tuple:
    """
    Routes to PageIndex or KG-FAISS based on doc_map availability.
    Returns (chunks, retrieval_method) where method is 'pageindex' or 'kg_faiss'.
    """
    # Find most recent PageIndex document for this ticker
    ticker_docs = {k: v for k, v in doc_map.items() if k.startswith(ticker + "_")}

    if ticker_docs:
        # Use most recent document (sort by accession number descending)
        best_key = sorted(ticker_docs.keys(), reverse=True)[0]
        accession = best_key.split("_")[1]
        candidates = [{"ticker": ticker, "accession": accession}]

        chunks = pageindex_retrieve(query, candidates, top_docs=1)
        if chunks:
            return chunks, "pageindex"

    # Fall back to KG-guided FAISS
    chunks = kg_guided_retrieve(query, ticker, top_k=3)
    return chunks, "kg_faiss"


def hybrid_answer(query: str, ticker: str = None) -> tuple:
    """
    Generates an answer using the hybrid retrieval strategy.
    Returns (answer, retrieval_method, chunks).
    """
    if ticker is None:
        ticker = detect_ticker_from_query(query)
    if ticker is None:
        # No ticker detected — fall back to vanilla FAISS
        chunks  = vanilla_rag_retrieve(query)
        context = "\n\n---\n\n".join([c["text"] for c in chunks])
        prompt  = (
            f"You are a financial analyst assistant. Answer the question using only "
            f"the context provided. Be precise with numbers.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {query}\n\nAnswer:"
        )
        return llm_chat(prompt), "vanilla_faiss", chunks

    chunks, method = hybrid_retrieve(query, ticker)

    if method == "pageindex":
        context = "\n\n---\n\n".join(
            [f"[{c['ticker']} | {c['title']}]\n{c['text']}" for c in chunks]
        )
    else:
        context = "\n\n---\n\n".join([c["text"] for c in chunks])

    prompt = (
        f"You are a financial analyst assistant. Answer the question using only "
        f"the context provided. Be precise with numbers.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\nAnswer:"
    )
    return llm_chat(prompt), method, chunks


# ------------------------------------------------------------------ updated consistency checker
def consistency_check(
    query:          str,
    context_ticker: str = None,
    context_year:   int = None,
    correct_output: bool = True,
) -> dict:
    """
    Full M3 pipeline using hybrid retrieval.
    """
    if context_ticker is None:
        context_ticker = detect_ticker_from_query(query)

    raw_answer, retrieval_method, chunks = hybrid_answer(query, context_ticker)

    claims          = extract_claims(raw_answer, context_ticker, context_year)
    verified_claims = verify_all_claims(claims)

    checkable  = [c for c in verified_claims if c["verdict"] != "UNVERIFIABLE"]
    halluc     = [c for c in checkable       if c["verdict"] == "HALLUCINATED"]

    hallucination_rate = len(halluc)    / len(checkable)       if checkable       else 0.0
    verification_rate  = len(checkable) / len(verified_claims) if verified_claims else 0.0

    verdicts = {"VERIFIED": 0, "HALLUCINATED": 0, "UNVERIFIABLE": 0}
    for c in verified_claims:
        verdicts[c["verdict"]] += 1

    final_answer = (
        build_corrected_answer(raw_answer, verified_claims)
        if correct_output else raw_answer
    )

    return {
        "query":              query,
        "raw_answer":         raw_answer,
        "final_answer":       final_answer,
        "retrieval_method":   retrieval_method,
        "claims":             claims,
        "verified_claims":    verified_claims,
        "hallucination_rate": hallucination_rate,
        "verification_rate":  verification_rate,
        "verdict_summary":    verdicts,
    }


# ------------------------------------------------------------------ smoke tests
print("Smoke test 1 — AAPL (should use PageIndex):")
ans, method, _ = hybrid_answer("What was Apple's total revenue in fiscal year 2024?", "AAPL")
print(f"  Method : {method}")
print(f"  Answer : {ans}\n")

print("Smoke test 2 — AMZN (should use KG-FAISS):")
ans, method, _ = hybrid_answer("What was Amazon's total revenue in fiscal year 2024?", "AMZN")
print(f"  Method : {method}")
print(f"  Answer : {ans}\n")

print("Smoke test 3 — NVDA (should use KG-FAISS):")
ans, method, _ = hybrid_answer("What was NVIDIA's total revenue in fiscal year 2025?", "NVDA")
print(f"  Method : {method}")
print(f"  Answer : {ans}")

## STEP 4: Expanded Evaluation — 20 queries across 4 companies

In [ ]:
# =============================================================================
# STEP 4: Expanded Evaluation — 20 queries across 4 companies
# Includes simple, medium, and hard queries to show system differentiation.
# =============================================================================

import pandas as pd

EVAL_QUERIES_FINAL = [
    # ── AAPL simple (PageIndex available) ──────────────────────────────
    {"query": "What was Apple's total revenue in fiscal year 2024?",
     "ticker": "AAPL", "year": 2024, "difficulty": "simple"},

    {"query": "What was Apple's net income in fiscal year 2024?",
     "ticker": "AAPL", "year": 2024, "difficulty": "simple"},

    {"query": "What was Apple's gross profit in fiscal year 2024?",
     "ticker": "AAPL", "year": 2024, "difficulty": "simple"},

    {"query": "What was Apple's operating income in fiscal year 2024?",
     "ticker": "AAPL", "year": 2024, "difficulty": "simple"},

    # ── AAPL harder (requires correct section navigation) ───────────────
    {"query": "What was Apple's total revenue in fiscal year 2022?",
     "ticker": "AAPL", "year": 2022, "difficulty": "medium"},

    {"query": "What was Apple's net income in fiscal year 2023?",
     "ticker": "AAPL", "year": 2023, "difficulty": "medium"},

    {"query": "What was Apple's gross profit in fiscal year 2023?",
     "ticker": "AAPL", "year": 2023, "difficulty": "medium"},

    # ── MSFT simple (PageIndex available) ──────────────────────────────
    {"query": "What was Microsoft's total revenue in fiscal year 2024?",
     "ticker": "MSFT", "year": 2024, "difficulty": "simple"},

    {"query": "What was Microsoft's net income in fiscal year 2024?",
     "ticker": "MSFT", "year": 2024, "difficulty": "simple"},

    {"query": "What was Microsoft's gross profit in fiscal year 2024?",
     "ticker": "MSFT", "year": 2024, "difficulty": "simple"},

    {"query": "What was Microsoft's operating income in fiscal year 2024?",
     "ticker": "MSFT", "year": 2024, "difficulty": "simple"},

    # ── MSFT harder ─────────────────────────────────────────────────────
    {"query": "What was Microsoft's total revenue in fiscal year 2023?",
     "ticker": "MSFT", "year": 2023, "difficulty": "medium"},

    {"query": "What was Microsoft's net income in fiscal year 2023?",
     "ticker": "MSFT", "year": 2023, "difficulty": "medium"},

    # ── AMZN (KG-FAISS path) ────────────────────────────────────────────
    {"query": "What was Amazon's total revenue in fiscal year 2024?",
     "ticker": "AMZN", "year": 2024, "difficulty": "simple"},

    {"query": "What was Amazon's net income in fiscal year 2024?",
     "ticker": "AMZN", "year": 2024, "difficulty": "simple"},

    {"query": "What was Amazon's gross profit in fiscal year 2024?",
     "ticker": "AMZN", "year": 2024, "difficulty": "simple"},

    {"query": "What was Amazon's operating income in fiscal year 2023?",
     "ticker": "AMZN", "year": 2023, "difficulty": "medium"},

    # ── NVDA (KG-FAISS path) ────────────────────────────────────────────
    {"query": "What was NVIDIA's total revenue in fiscal year 2025?",
     "ticker": "NVDA", "year": 2025, "difficulty": "simple"},

    {"query": "What was NVIDIA's net income in fiscal year 2025?",
     "ticker": "NVDA", "year": 2025, "difficulty": "simple"},

    {"query": "What was NVIDIA's gross profit in fiscal year 2024?",
     "ticker": "NVDA", "year": 2024, "difficulty": "medium"},
]


def eval_single(query: str, ticker: str, year: int, answer: str) -> dict:
    claims  = extract_claims(answer, context_ticker=ticker, context_year=year)
    vclaims = verify_all_claims(claims)

    checkable = [c for c in vclaims if c["verdict"] != "UNVERIFIABLE"]
    halluc    = [c for c in checkable if c["verdict"] == "HALLUCINATED"]

    return {
        "n_claims":           len(vclaims),
        "hallucination_rate": len(halluc)    / len(checkable) if checkable else 0.0,
        "verification_rate":  len(checkable) / len(vclaims)   if vclaims   else 0.0,
        "verdict_summary": {
            "VERIFIED":     sum(1 for c in vclaims if c["verdict"] == "VERIFIED"),
            "HALLUCINATED": sum(1 for c in vclaims if c["verdict"] == "HALLUCINATED"),
            "UNVERIFIABLE": sum(1 for c in vclaims if c["verdict"] == "UNVERIFIABLE"),
        }
    }


def run_vanilla_eval(item: dict) -> dict:
    answer = vanilla_rag_answer(item["query"])
    stats  = eval_single(item["query"], item["ticker"], item["year"], answer)
    return {
        "system":     "Vanilla RAG",
        "query":      item["query"],
        "ticker":     item["ticker"],
        "year":       item["year"],
        "difficulty": item["difficulty"],
        "answer":     answer,
        "retrieval":  "faiss_global",
        **stats
    }


def run_hybrid_eval(item: dict) -> dict:
    answer, method, _ = hybrid_answer(item["query"], item["ticker"])
    stats  = eval_single(item["query"], item["ticker"], item["year"], answer)
    return {
        "system":     "KG+PageIndex RAG",
        "query":      item["query"],
        "ticker":     item["ticker"],
        "year":       item["year"],
        "difficulty": item["difficulty"],
        "answer":     answer,
        "retrieval":  method,
        **stats
    }


def run_reasonkg_eval(item: dict) -> dict:
    result = consistency_check(
        query          = item["query"],
        context_ticker = item["ticker"],
        context_year   = item["year"],
        correct_output = True,
    )
    stats = eval_single(
        item["query"], item["ticker"], item["year"], result["final_answer"]
    )
    return {
        "system":     "ReasonKG",
        "query":      item["query"],
        "ticker":     item["ticker"],
        "year":       item["year"],
        "difficulty": item["difficulty"],
        "answer":     result["final_answer"],
        "retrieval":  result["retrieval_method"],
        **stats
    }


# ── Run evaluation ───────────────────────────────────────────────────────────
print(f"Running evaluation: {len(EVAL_QUERIES_FINAL)} queries x 3 systems "
      f"= {len(EVAL_QUERIES_FINAL)*3} total API calls\n")

all_results = []

for i, item in enumerate(EVAL_QUERIES_FINAL):
    print(f"[{i+1:02d}/{len(EVAL_QUERIES_FINAL)}] [{item['difficulty'].upper():6s}] "
          f"{item['ticker']} — {item['query'][:55]}...")

    v  = run_vanilla_eval(item)
    h  = run_hybrid_eval(item)
    rk = run_reasonkg_eval(item)

    all_results.extend([v, h, rk])

    print(f"         Vanilla  : hall={v['hallucination_rate']*100:.0f}%  "
          f"claims={v['n_claims']}  method={v['retrieval']}")
    print(f"         KG+PI    : hall={h['hallucination_rate']*100:.0f}%  "
          f"claims={h['n_claims']}  method={h['retrieval']}")
    print(f"         ReasonKG : hall={rk['hallucination_rate']*100:.0f}%  "
          f"claims={rk['n_claims']}  method={rk['retrieval']}")
    print()

    time.sleep(0.5)


# ── Summary tables ───────────────────────────────────────────────────────────
df_results = pd.DataFrame(all_results)

# Overall summary
SYSTEM_ORDER = ["Vanilla RAG", "KG+PageIndex RAG", "ReasonKG"]
df_summary = (
    df_results
    .groupby("system")[["hallucination_rate", "verification_rate", "n_claims"]]
    .mean()
    .round(4)
    .reset_index()
)
df_summary["hallucination_pct"] = (df_summary["hallucination_rate"] * 100).round(1)
df_summary["verification_pct"]  = (df_summary["verification_rate"]  * 100).round(1)
df_summary["system"] = pd.Categorical(df_summary["system"], categories=SYSTEM_ORDER, ordered=True)
df_summary = df_summary.sort_values("system").reset_index(drop=True)

print("\n=== Overall Evaluation Summary ===")
print(df_summary[["system", "hallucination_pct", "verification_pct", "n_claims"]].to_string(index=False))

# By difficulty
df_diff = (
    df_results
    .groupby(["system", "difficulty"])["hallucination_rate"]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
    .pivot(index="system", columns="difficulty", values="hallucination_rate")
    .reindex(SYSTEM_ORDER)
)
print("\n=== Hallucination Rate by Difficulty ===")
print(df_diff.to_string())

# By ticker
df_ticker = (
    df_results
    .groupby(["system", "ticker"])["hallucination_rate"]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
    .pivot(index="system", columns="ticker", values="hallucination_rate")
    .reindex(SYSTEM_ORDER)
)
print("\n=== Hallucination Rate by Company ===")
print(df_ticker.to_string())

# Save
df_results.to_csv("/content/drive/MyDrive/eval_results_final.csv",   index=False)
df_summary.to_csv("/content/drive/MyDrive/eval_summary_final.csv",   index=False)
print("\nResults saved to Google Drive.")

## STEP 5: Final Presentation Visualizations

In [ ]:
# =============================================================================
# STEP 5: Final Presentation Visualizations
# Produces 5 publication-ready figures saved to Google Drive.
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np
import warnings
warnings.filterwarnings("ignore")

SYSTEM_ORDER  = ["Vanilla RAG", "KG+PageIndex RAG", "ReasonKG"]
SYSTEM_COLORS = ["#C0392B", "#2980B9", "#27AE60"]
SAVE_DIR      = "/content/drive/MyDrive"

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         11,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
})

halluc_vals = [
    df_summary.loc[df_summary["system"] == s, "hallucination_pct"].values[0]
    for s in SYSTEM_ORDER
]
verif_vals = [
    df_summary.loc[df_summary["system"] == s, "verification_pct"].values[0]
    for s in SYSTEM_ORDER
]


# ------------------------------------------------------------------ Figure 1
# Overall hallucination rate
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(SYSTEM_ORDER, halluc_vals, color=SYSTEM_COLORS, width=0.5,
              edgecolor="white", linewidth=1.2)
ax.set_ylabel("Hallucination Rate (%)", fontsize=12)
ax.set_title("Hallucination Rate by System\n(lower is better)",
             fontsize=13, fontweight="bold")
ax.set_ylim(0, max(halluc_vals) * 1.5 + 2)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
for bar, val in zip(bars, halluc_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=12, fontweight="bold")
# Improvement annotation
ax.annotate(
    "90% reduction\nvs Vanilla RAG",
    xy=(2, halluc_vals[2]),
    xytext=(1.5, halluc_vals[0] * 0.6),
    arrowprops=dict(arrowstyle="->", color="black", lw=1.5),
    fontsize=9, color="black",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="gray")
)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/fig1_hallucination_rate.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 1 saved.")


# ------------------------------------------------------------------ Figure 2
# Hallucination by difficulty
diff_data = df_diff.reindex(SYSTEM_ORDER)
x     = np.arange(len(SYSTEM_ORDER))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, diff_data["simple"].fillna(0), width,
               label="Simple queries", color=[c + "cc" for c in ["#C0392B","#2980B9","#27AE60"]],
               edgecolor="white")
bars2 = ax.bar(x + width/2, diff_data["medium"].fillna(0), width,
               label="Medium queries", color=SYSTEM_COLORS, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(SYSTEM_ORDER, fontsize=10)
ax.set_ylabel("Hallucination Rate (%)", fontsize=12)
ax.set_title("Hallucination Rate by Query Difficulty\n(lower is better)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.set_ylim(0, max(diff_data.max()) * 1.4 + 3)

for bar in bars1:
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{bar.get_height():.1f}%", ha="center", fontsize=9, fontweight="bold")
    else:
        ax.text(bar.get_x() + bar.get_width()/2, 0.5,
                "0%", ha="center", fontsize=9, color="green", fontweight="bold")

for bar in bars2:
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{bar.get_height():.1f}%", ha="center", fontsize=9, fontweight="bold")
    else:
        ax.text(bar.get_x() + bar.get_width()/2, 0.5,
                "0%", ha="center", fontsize=9, color="green", fontweight="bold")

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/fig2_by_difficulty.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 2 saved.")


# ------------------------------------------------------------------ Figure 3
# Hallucination by company
ticker_data = df_ticker.reindex(SYSTEM_ORDER)
tickers     = ticker_data.columns.tolist()
x           = np.arange(len(tickers))
width       = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
for i, (system, color) in enumerate(zip(SYSTEM_ORDER, SYSTEM_COLORS)):
    vals = ticker_data.loc[system].fillna(0).values
    offset = (i - 1) * width
    bars = ax.bar(x + offset, vals, width, label=system,
                  color=color, edgecolor="white", linewidth=1)
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    f"{val:.0f}%", ha="center", fontsize=8, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(tickers, fontsize=11)
ax.set_ylabel("Hallucination Rate (%)", fontsize=12)
ax.set_title("Hallucination Rate by Company\n(lower is better)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.set_ylim(0, max(ticker_data.max()) * 1.5 + 3)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/fig3_by_company.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 3 saved.")


# ------------------------------------------------------------------ Figure 4
# Per-query heatmap
df_pivot = df_results.pivot_table(
    index="query", columns="system", values="hallucination_rate"
)[SYSTEM_ORDER]

short_labels = [
    q.replace("What was ", "").replace(" in fiscal year", " FY").replace("?", "")
    for q in df_pivot.index
]
df_pivot.index = short_labels

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(df_pivot.values, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=1)

ax.set_xticks(range(len(SYSTEM_ORDER)))
ax.set_xticklabels(SYSTEM_ORDER, fontsize=11, fontweight="bold")
ax.set_yticks(range(len(df_pivot.index)))
ax.set_yticklabels(df_pivot.index, fontsize=8)

for i in range(len(df_pivot.index)):
    for j in range(len(SYSTEM_ORDER)):
        val   = df_pivot.values[i, j]
        label = "HALL." if val > 0 else "OK"
        color = "white" if val > 0.5 else "black"
        ax.text(j, i, label, ha="center", va="center",
                fontsize=9, color=color, fontweight="bold")

cbar = plt.colorbar(im, ax=ax, shrink=0.7)
cbar.set_label("Hallucination Rate", fontsize=10)
ax.set_title("Per-Query Hallucination Heatmap — ReasonKG CSE 579 Group 6\n"
             "(green = correct, red = hallucinated)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/fig4_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 4 saved.")


# ------------------------------------------------------------------ Figure 5
# Summary dashboard — all key metrics in one slide-ready figure
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("ReasonKG Evaluation Summary — CSE 579 Group 6",
             fontsize=14, fontweight="bold", y=1.02)

# Panel 1: hallucination rate
axes[0].bar(SYSTEM_ORDER, halluc_vals, color=SYSTEM_COLORS,
            width=0.5, edgecolor="white")
axes[0].set_title("Hallucination Rate\n(lower is better)", fontweight="bold")
axes[0].set_ylabel("Rate (%)")
axes[0].set_ylim(0, max(halluc_vals) * 1.6 + 3)
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)
for i, val in enumerate(halluc_vals):
    axes[0].text(i, val + 0.3, f"{val:.1f}%",
                 ha="center", fontsize=10, fontweight="bold")

# Panel 2: verification coverage
axes[1].bar(SYSTEM_ORDER, verif_vals, color=SYSTEM_COLORS,
            width=0.5, edgecolor="white")
axes[1].set_title("KG Verification Coverage\n(higher is better)", fontweight="bold")
axes[1].set_ylabel("Rate (%)")
axes[1].set_ylim(0, 120)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)
for i, val in enumerate(verif_vals):
    axes[1].text(i, val + 0.8, f"{val:.1f}%",
                 ha="center", fontsize=10, fontweight="bold")

# Panel 3: claims extracted per query
n_claims_vals = [
    df_summary.loc[df_summary["system"] == s, "n_claims"].values[0]
    for s in SYSTEM_ORDER
]
axes[2].bar(SYSTEM_ORDER, n_claims_vals, color=SYSTEM_COLORS,
            width=0.5, edgecolor="white")
axes[2].set_title("Avg Claims Extracted\nper Query", fontweight="bold")
axes[2].set_ylabel("Count")
axes[2].set_ylim(0, max(n_claims_vals) * 1.4 + 0.3)
axes[2].spines["top"].set_visible(False)
axes[2].spines["right"].set_visible(False)
for i, val in enumerate(n_claims_vals):
    axes[2].text(i, val + 0.02, f"{val:.2f}",
                 ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/fig5_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 5 saved.")


# ------------------------------------------------------------------ Print key stats
print("\n=== Key Statistics for Presentation ===")
print(f"Total queries evaluated  : {len(EVAL_QUERIES_FINAL)}")
print(f"Companies covered        : AAPL, MSFT, AMZN, NVDA")
print(f"Fiscal years covered     : 2022 - 2025")
print(f"KG metrics in Neo4j      : {len(store.query_all_metrics('AAPL')) + len(store.query_all_metrics('MSFT')) + len(store.query_all_metrics('AMZN')) + len(store.query_all_metrics('NVDA'))} total records")
print(f"SEC filings indexed      : 26 (FAISS) + 2 (PageIndex)")
print(f"Retrieval methods used   : PageIndex (AAPL/MSFT), KG-FAISS (AMZN/NVDA)")
print(f"\nHallucination reduction  :")
print(f"  Vanilla RAG -> ReasonKG : {halluc_vals[0]:.1f}% -> {halluc_vals[2]:.1f}% "
      f"({((halluc_vals[0]-halluc_vals[2])/halluc_vals[0]*100):.0f}% reduction)")
print(f"\nAll figures saved to: {SAVE_DIR}")